In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

#### Let's Implement a Binary Class Classifier Decision Tree Class. 

In [17]:
# Define the node class 
class TreeNode: 
    def __init__(self , gini , num_samples , num_pos_class , predicted_class): 
        self.gini = gini # Gini -> Impurity of the node. Formula: 1 - sum i=1 to K p(i)**2
        self.num_samples = num_samples # total samples in this node
        self.num_pos_class = num_pos_class # Number of positive class samples(1s)
        self.predicted_class = predicted_class # Majority class label(0 or 1)

        self.feature_index = None # which feature to split on
        self.threshold = None # Split Value(for numerical) or category(for categorical)
        self.left = None # Left child
        self.right = None # Right Child

In [56]:
# Make the Decision-tree class 
class MyDecisionTreeBinaryClassifier: 
    def __init__(self , max_depth = None , min_samples_split = 2): 
        self.max_depth = max_depth
        # min_samples_split: Minimum number of samples required to split a node. 
        # If a node contains fewer than min_samples_split samples, it will not be split even if it is impure. It becomes Leaf Node
        self.min_samples_split = min_samples_split
        self.root = None # main root of the tree
        self.feature_types = [] # 'numerical' or 'categorical'

    def fit(self , X , y): 
        # Store the feature types 
        if isinstance(X , pd.DataFrame): 
            self.feature_names = X.columns.tolist()
            X = X.values
        if isinstance(y , pd.Series): 
            y = y.values

        # Detect feature types (simple check: if dtype is object or str, assume categorical)
        for col in X.T: 
            if isinstance(col[0] , str) or isinstance(col[0] , object): 
                self.feature_types.append('categorical')
            else:
                self.feature_types.append('numerical') 

        # store the target classes
        self.n_classes = len(set(y))
        self.root = self._build_tree(X , y , depth = 0)

    def predict(self , X): 
        if isinstance(X , pd.DataFrame): 
            X = X.to_numpy()
        return np.array([self._predict_row(row , self.root) for row in X])


    # Find gini 
    def _gini(self , y): 
        '''Calculate Gini Impurity for Binary classification'''
        m = len(y) 
        if m == 0: 
            return 0
        # Find the probability for class 1 and 0 
        p1 = np.sum(y) / m # If class label is 1 then count of class 1 will be sum of y and probability = sum(y)/m 
        p0 = 1 - p1 
        gini = 1 - ((p1 ** 2) + (p0 ** 2))
        return gini

    # Build the tree and return the root 
    def _build_tree(self , X , y , depth): 
        num_samples = len(y) # total rows
        num_pos = np.sum(y) # count of class 1
        # majority class will be predicted class 
        predicted_class = 1 if num_pos >= (num_samples / 2) else 0
        
        # Create the node 
        node = TreeNode(
            gini = self._gini(y), 
            num_samples = num_samples, 
            num_pos_class = num_pos,
            predicted_class = predicted_class
        )

        # Stopping criteria 
        if (depth >= self.max_depth if self.max_depth is not None else False) or num_samples < self.min_samples_split or node.gini == 0: 
            # stop here and return the node  
            return node

        # otherwise try to find the best split 
        idx , thr = self._best_split(X , y) 

        if idx is None: # No valid split 
            return node 

        # Otherwise do the split 
        node.feature_index = idx
        # also set the tresold value 
        node.threshold = thr

        # if split column is numerical 
        if self.feature_types[idx] == 'numerical': 
            left_idx = X[ : , idx] <= thr 
        else: # categorical
            left_idx = X[ : , idx] == thr 

        X_left , y_left = X[left_idx] , y[left_idx]
        # right will be remaining after left_idx 
        X_right , y_right = X[~left_idx] , y[~left_idx]

        node.left = self._build_tree(X_left , y_left , depth + 1)
        node.right = self._build_tree(X_right , y_right , depth + 1)

        return node 

    # make the best split function 
    def _best_split(self , X , y):
        best_gini = 1.0
        best_idx , best_thr = None , None

        # Go the every column of X 
        for idx in range(X.shape[1]): 
            values = X[ : , idx] # all values of current col 
            feature_type = self.feature_types[idx]

            if feature_type == 'numerical': 
                gini , threshold = self._best_split_numerical(values , y) 
            else:
                gini , threshold = self._best_split_categorical(values , y)

            # we want the lowest gini 
            if gini < best_gini: 
                best_gini = gini
                best_idx = idx
                best_thr = threshold

        return best_idx , best_thr

    def _best_split_numerical(self , values , y):
        """Find the best thresold to split a numerical feature."""
        best_gini = 1.0
        best_threshold = None 

        thresholds = np.unique(values)
        for t in thresholds: 
            # try to split at t value 
            left = y[values <= t] 
            right = y[values > t] 

            if len(left) == 0 or len(right) == 0: 
                # in case of pure split do nothing 
                continue
            n_left , n_right = len(left) , len(right) 
            gini_left = self._gini(left)
            gini_right = self._gini(right)

            # weighted avg gini
            n_total = n_left + n_right
            gini = ((n_left / n_total) * gini_left) + ((n_right / n_total) * gini_right)
            if gini < best_gini: 
                best_gini = gini
                best_threshold = t 
        return best_gini , best_threshold
    def _best_split_categorical(self , values , y):
        """Find the best thresold to split a categorical feature."""
        best_gini = 1.0 
        best_category = None 

        categories = np.unique(values)
        for cat in categories:
            # based on cat split left and right 
            left = y[values == cat] 
            right = y[values != cat]
            if len(left) == 0 or len(right) == 0: 
                # in case of pure split do nothing 
                continue 
            n_left , n_right = len(left) , len(right) 
            gini_left = self._gini(left)
            gini_right = self._gini(right)
            n_total = n_left + n_right

            gini = ((n_left / n_total) * gini_left) + ((n_right / n_total) * gini_right) 
            if gini < best_gini: 
                best_gini = gini
                best_category = cat
                
        return best_gini , best_category

    def _is_leaf(self , node): 
        return node.left is None and node.right is None
        
    def _predict_row(self , row , node): 
        if self._is_leaf(node): 
            # return the current node predicted class 
            return node.predicted_class

        val = row[node.feature_index]

        # ask question on best_split feature 
        # if numerical feature 
        if self.feature_types[node.feature_index] == 'numerical': 
            if val <= node.threshold: 
                return self._predict_row(row , node.left)
            else:
                return self._predict_row(row , node.right)
        else: # categorical
           if val == node.threshold: 
               return self._predict_row(row , node.left)
           else:
               return self._predict_row(row , node.right)    

    def print_tree(self , node = None , spacing = ""): 
        if node is None:
           node = self.root
        # If it's a leaf node, print the prediction
        if node.left is None and node.right is None:
            print(spacing + f"Predict: {node.predicted_class} (samples: {node.num_samples}, gini: {node.gini:.3f})")
            return
        # Print the decision rule
        feature_name = self.feature_names[node.feature_index] if hasattr(self, 'feature_names') else f"feature[{node.feature_index}]"
        if self.feature_types[node.feature_index] == 'numerical':
            print(spacing + f"[{feature_name} <= {node.threshold}]")
        else:
            print(spacing + f"[{feature_name} == '{node.threshold}']")
    
        # Recursively print left and right branches
        print(spacing + '--> True:')
        self.print_tree(node.left, spacing + "    ")
    
        print(spacing + '--> False:')
        self.print_tree(node.right, spacing + "    ")

In [57]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [58]:
# Load and prepare data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

In [59]:
# Select 3 numerical features for simplicity
X = X[['mean radius', 'mean texture', 'mean perimeter']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [60]:
tree = MyDecisionTreeBinaryClassifier(max_depth=4, min_samples_split=5)
tree.fit(X_train, y_train)

In [61]:
y_pred = tree.predict(X_test)

In [62]:
y_pred.shape

(114,)

In [63]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6140350877192983


In [64]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=4, min_samples_split=5, random_state=42)
clf.fit(X_train, y_train)
y_pred_sklearn = clf.predict(X_test)

print("Sklearn Accuracy:", accuracy_score(y_test, y_pred_sklearn))

Sklearn Accuracy: 0.9122807017543859
